In [1]:
print("Hello, World!")

Hello, World!


In [ ]:
%pwd

'c:\\AI&ML\\my Projects\\Medical_chatbot\\Medical-Chatbot-with-LLMs-LangChain-Pinecone-Flask-AWS-\\research'

In [2]:
import os

In [3]:
os.chdir("../")

In [6]:
%pwd

'c:\\AI&ML\\my Projects\\Medical_chatbot\\Medical-Chatbot-with-LLMs-LangChain-Pinecone-Flask-AWS-'

In [4]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\VICTUS\miniconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
#extract text from pdf file

def load_pdf_files(file):
    loader = DirectoryLoader(
        file,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    document = loader.load()
    return document

In [6]:
extracted_data = load_pdf_files("data")

In [7]:
extracted_data[0]

Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Medical_book.pdf', 'total_pages': 637, 'page': 0, 'page_label': '1'}, page_content='')

In [7]:
len(extracted_data)


637

In [8]:
from typing import List
from langchain.schema import Document

In [9]:
#filtering the document removed unwanted things and keeps useful data its a cleaning process
def filter_doc(docs: List[Document]) -> List[Document]:
    filtered_docs: List[Document] = []
    for doc in docs:
        stc = doc.metadata.get("source")
        filtered_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": stc}
            )
        )
    return filtered_docs

In [10]:
minimal_docs = filter_doc(extracted_data)

In [11]:
minimal_docs[:5]

[Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content=''),
 Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1'),
 Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='STAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate Editor\nKate Kretschmann, Melissa C. McDade, Ryan\nThomason, Assistant Editors\nMark Springer, Technical Specialist\nAndrea Lopeman, Programmer/Analyst\nBarbara J. Yarrow,Manager, Imaging and Multimedia\nContent\nRobyn V . Young,Project Manager, Imaging and\nMultimedia Content\nDean Dauphinais, Senior Editor, Imaging and\nMultimed

In [11]:
# splitting the document into smaller chunks
def test_split(minimal_docs):
    text_spliter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=len
    )
    chunk = text_spliter.split_documents(minimal_docs)
    return chunk


In [12]:
test_chunk = test_split(minimal_docs)
print(len(test_chunk))

5961


In [13]:
from langchain.embeddings import HuggingFaceEmbeddings
import torch

In [14]:
#downloading the embedding model and creating the embedding object
def download_embedding():
    model_name = "BAAI/bge-small-en-v1.5"
    embeddings = HuggingFaceEmbeddings(model_name=model_name,
                                       model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"}
                                       )
    return embeddings


In [15]:
embeding = download_embedding()

C:\Users\VICTUS\AppData\Local\Temp\ipykernel_22936\730432546.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name,
c:\Users\VICTUS\miniconda3\envs\medibot\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [16]:
vector = embeding.embed_query("hello world")

In [91]:
len(vector)

384

In [20]:
len(vector)

384

In [17]:
from dotenv import load_dotenv
import os
import google.generativeai as genai
load_dotenv()

c:\Users\VICTUS\miniconda3\envs\medibot\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


True

In [18]:
pinecone_api_key = os.getenv("pinecone_api_key")
# gemini_api_key = os.getenv("gemini_api_key")

In [19]:
os.environ["pinecone_api_key"] = pinecone_api_key
# os.environ["gemini_api_key"] = gemini_api_key
# gemini_key = gemini_api_key

In [20]:
from pinecone import Pinecone
pinecone_key = pinecone_api_key
pc = Pinecone(pinecone_key)

In [24]:
pc

In [ ]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot-v2"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore
doc = PineconeVectorStore.from_documents(
    documents=test_chunk,
    embedding=embeding,
    index_name=index_name
)

In [21]:
# This for searching the document in the already created index
index_name = "medical-chatbot"
from langchain_pinecone import PineconeVectorStore


docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeding

)

In [22]:
retriever = docsearch.as_retriever(search_type = "similarity", search_kwargs={"k": 1})

In [23]:
retrieved_docs = retriever.invoke("what is cancer?")
print(retrieved_docs)

[Document(id='0f536440-af74-4b44-8db6-e47390095e1f', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Leukemia —A cancer of blood cells characterized\nby the abnormal increase in the number of white\nblood cells in the tissues. There are many types of\nleukemias and they are classified according to the\ntype of white blood cell involved.\nLymphoma—A blood cancer in which lymphocytes,\na variety of white blood cells, grow at an unusually\nrapid rate.\nMutation—Any change in the hereditary material of\ngenes.\nPurkinje’s cells—Large branching cells of the ner-\nvous system.')]


In [23]:
retrieved_docs

[Document(id='0f536440-af74-4b44-8db6-e47390095e1f', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Leukemia —A cancer of blood cells characterized\nby the abnormal increase in the number of white\nblood cells in the tissues. There are many types of\nleukemias and they are classified according to the\ntype of white blood cell involved.\nLymphoma—A blood cancer in which lymphocytes,\na variety of white blood cells, grow at an unusually\nrapid rate.\nMutation—Any change in the hereditary material of\ngenes.\nPurkinje’s cells—Large branching cells of the ner-\nvous system.')]

In [24]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline

In [27]:
model_id = "gpt2"

In [28]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

c:\Users\VICTUS\miniconda3\envs\medibot\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VICTUS\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\VICTUS\miniconda3\envs\medibot\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_downl

In [29]:

pipe = pipeline(
    task="text-generation",     # task type
    model=model,                # your loaded model
    tokenizer=tokenizer,        # your loaded tokenizer
    max_new_tokens=60,          # how long the response should be
    temperature=0.7,            # randomness
    do_sample=True,             # use sampling instead of greedy decoding
    return_full_text=False      # only return the generated part
)

from langchain import HuggingFacePipeline

chatmodel = HuggingFacePipeline(pipeline=pipe)

C:\Users\VICTUS\AppData\Local\Temp\ipykernel_22936\2655516630.py:13: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  chatmodel = HuggingFacePipeline(pipeline=pipe)


In [30]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [31]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = "You are a medical assistant. Answer the question using only the provided context. Do not use your own knowledge. Provide a direct answer in one sentence."

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Context:\n{context}\n\nQuestion: {input}\n\nAnswer:")
])

In [32]:
question_answer_chain = create_stuff_documents_chain(chatmodel,prompt=prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [35]:
response = rag_chain.invoke({"input": "what is cancer?"})
print(response["answer"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


 The body contains many cells. A certain type of
cells is called a cancer. The body produces many cells for more

than the number of cells in one body. In cancers, only a certain number of

cells are produced.

There are about 1,000 different types of


cuda:0


c:\Users\VICTUS\miniconda3\envs\medibot\lib\site-packages\transformers\generation\configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


 The term "Cancer" refers to diseases characterized by abnormal cellular proliferation resulting from genetic mutations within those cells
